# # 종합 실습 가이드_모델개발 및 최적화



- 실습 주제 : 머신러닝 기법을 활용한 은행 고객 이탈 예측 모형 생성
- 실습 목표 : Validation 및 Test 데이터의 평균 AUC 최대화
- 실습 조건
  - 분석 데이타 : 실습에서 사용한 동일한 데이타 (bank_churn_train.csv)
  - Target 변수 : Exited (이탈 여부) *Target 변수는 어떠한 데이타 변환도 하지 않음
  - 학습 조건 :  아래 Baseline Code 를 참고하여 **(필수)** 과정인 데이타 불러오기 및 데이터 분할만 **아래 제시된 코드를 그대로 사용**하고, 그외 모든 과정은 자율적으로 선택
  - 데이타 건수 : 데이타 제거는 하지 않음 (전체/Train/Test 건수 그대로 유지), 중복 데이타가 있어도 제거하지 않음
  - 기타 : 수업에서 다루지 않은 분석 기법 사용도 허용. 예) AutoML
- 실습 제출물
   - 종합실습_결과물_모델개발및최적화_OOO.xlsx 에 실습 결과 작성 후 코드 (.ipynb)와 같이 제출   
   - 실습 결과는 제출 코드로 재현이 가능해야 함
   - 제출시 화일명은 본인 이름으로 2개 화일 제출 (엑셀, 코드)   
     예) **종합실습_결과물_모델개발및최적화_홍길동.xlsx**, **종합실습_결과물_모델개발및최적화_홍길동.ipynb**


# Baseline Code
데이타 불러오기 **(필수)**

In [161]:
import pandas as pd

df = pd.read_csv("bank_churn_train.csv", encoding="cp949")

Data Partition (6:2:2) **(필수)**



In [162]:
from sklearn.model_selection import train_test_split

# 1단계: train 60% / (valid + test) 40%
df_train, temp = train_test_split(df, test_size=0.4, random_state=42)

# 2단계: temp를 valid 50% / test 50% → 전체 기준 각 20%
df_valid, df_test = train_test_split(temp, test_size=0.5, random_state=42)

X, Y 분리

In [163]:
# Train
X_train = df_train.drop('Exited', axis=1)
Y_train = df_train['Exited']

# Valid
X_valid = df_valid.drop('Exited', axis=1)
Y_valid = df_valid['Exited']

# Test
X_test = df_test.drop('Exited', axis=1)
Y_test = df_test['Exited']

One-Hot Encoding

In [164]:
import pandas as pd

# 범주형 컬럼 선택 (train 기준)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# One-Hot Encoding
X_train = pd.get_dummies(X_train, drop_first=False, dummy_na=False)
X_valid = pd.get_dummies(X_valid, drop_first=False, dummy_na=False)
X_test  = pd.get_dummies(X_test,  drop_first=False, dummy_na=False)

# valid/test 를 train 컬럼 구조에 맞추기
#    - train에 있고, valid/test에 없는 컬럼 → 0으로 채움
#    - train에 없고, valid/test에 있는 컬럼 → 제거
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test  = X_test.reindex(columns=X_train.columns,  fill_value=0)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test  shape:", X_test.shape)

X_train shape: (1260, 1953)
X_valid shape: (420, 1953)
X_test  shape: (420, 1953)


/var/folders/14/gczk6h2n7nb_w800tdh9mtb40000gn/T/ipykernel_81423/2967721449.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


학습 및 평가

In [165]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             precision_score, roc_auc_score)

# 모델 생성
model = DecisionTreeClassifier(random_state=42, max_depth=3)
model.fit(X_train, Y_train)

# X_valid 예측
Y_valid_pred = model.predict(X_valid)
Y_valid_prob = model.predict_proba(X_valid)[:, 1]  # AUC용 확률값

# X_test 예측
Y_test_pred = model.predict(X_test)
Y_test_prob = model.predict_proba(X_test)[:, 1]    # AUC용 확률값

# 평가 함수 정의
def evaluate(Y_actual, Y_pred, Y_prob, dataset_name):
    accuracy  = accuracy_score(Y_actual, Y_pred)
    f1        = f1_score(Y_actual, Y_pred)
    recall    = recall_score(Y_actual, Y_pred)
    precision = precision_score(Y_actual, Y_pred)
    auc       = roc_auc_score(Y_actual, Y_prob)

    print(f"\n=== {dataset_name} 평가 ===")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"AUC      : {auc:.4f}")

    return {'Accuracy': accuracy, 'F1': f1, 'Recall': recall,
            'Precision': precision, 'AUC': auc}

# Validation, Test 평가 실행
valid_scores = evaluate(Y_valid, Y_valid_pred, Y_valid_prob, "Validation")
test_scores  = evaluate(Y_test,  Y_test_pred,  Y_test_prob,  "Test")


=== Validation 평가 ===
Accuracy : 0.7833
F1 Score : 0.5285
Recall   : 0.4811
Precision: 0.5862
AUC      : 0.7820

=== Test 평가 ===
Accuracy : 0.8476
F1 Score : 0.4754
Recall   : 0.4265
Precision: 0.5370
AUC      : 0.7625


# 실습 시작

In [166]:
import pandas as pd

df = pd.read_csv("bank_churn_train.csv", encoding="cp949")

# Feature Creation

In [167]:
df["baseDate"] = pd.to_datetime(df["baseDate"])                       # 문자열 -> 날짜 type으로 변환
df["accountOpeningDate"] = pd.to_datetime(df["accountOpeningDate"])   # 문자열 -> 날짜 type으로 변환

# 계좌 유지 기간
df["TenureDays"] = (df["baseDate"] - df["accountOpeningDate"]).dt.days
df["TenureDays"] = df["TenureDays"] / 365.25

# 상품당 평균 잔액
df["BalancePerProduct"] = (df["Balance"] / df["NumOfProducts"].replace(0, np.nan))

# 급여 대비 잔액 비율
df["BalanceSalaryRatio"] = (df["Balance"] / df["EstimatedSalary"].replace(0, np.nan))

In [168]:
# 불필요 변수 제거
df = df.drop(columns=["Surname", "baseDate", "accountOpeningDate"])

# EDA

In [169]:
df.select_dtypes(include=[np.number]).describe()

,CustomerId,CreditScore,Age,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,TenureDays,BalancePerProduct,BalanceSalaryRatio
count,2.100000e+03,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000,2031.000000,2100.000000,2100.000000,2100.000000,2031.000000
mean,1.568984e+07,649.784762,38.813333,76605.061957,1.536667,0.715238,0.514286,100611.165101,0.213333,5.541430,62662.084541,2.899178
std,7.211313e+04,96.818063,10.231034,62865.409749,0.591417,0.451408,0.499915,57715.705901,0.409759,2.901019,57167.384588,15.563732
min,1.556570e+07,350.000000,18.000000,0.000000,1.000000,0.000000,0.000000,216.270000,0.000000,0.002738,0.000000,0.000000
25%,1.562694e+07,580.000000,32.000000,0.000000,1.000000,0.000000,0.000000,51680.575000,0.000000,3.061602,0.000000,0.000000
50%,1.568962e+07,652.000000,37.000000,97535.745000,1.000000,1.000000,1.000000,101057.950000,0.000000,5.500342,59513.007500,0.756306
75%,1.575285e+07,717.000000,44.000000,128738.242500,2.000000,1.000000,1.000000,149756.130000,0.000000,8.042437,112044.957500,1.539279
max,1.581556e+07,850.000000,88.000000,250898.090000,4.000000,1.000000,1.000000,199857.470000,1.000000,10.989733,213146.200000,349.521987


In [170]:
# 범주형 변수 자동 추출
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# 범주 건수 & 비율 계산
for col in cat_cols:
    print(f"\n[{col}]")
    dist = df[col].value_counts(dropna=False)
    ratio = df[col].value_counts(normalize=True, dropna=False) * 100
    result = pd.DataFrame({
        "count": dist,
        "ratio_%": ratio.round(2)
    })
    print(result)


[Geography]
           count  ratio_%
Geography                
France      1028    48.95
Spain        528    25.14
Germany      502    23.90
NaN           42     2.00

[Gender]
        count  ratio_%
Gender                
Male     1116    53.14
Female    984    46.86


/var/folders/14/gczk6h2n7nb_w800tdh9mtb40000gn/T/ipykernel_81423/2971682825.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns


# 데이터 분할

In [171]:
from sklearn.model_selection import train_test_split

# 1단계: train 60% / (valid + test) 40%
df_train, temp = train_test_split(df, test_size=0.4, random_state=42)

# 2단계: temp를 valid 50% / test 50% → 전체 기준 각 20%
df_valid, df_test = train_test_split(temp, test_size=0.5, random_state=42)

# 원본 데이터 복사
df_train_raw = df_train.copy()
df_valid_raw  = df_valid.copy()
df_test_raw  = df_test.copy()

# 분석 변수 선택
df_train = df_train.drop(columns=["CustomerId"])
df_valid = df_valid.drop(columns=["CustomerId"])
df_test = df_test.drop(columns=["CustomerId"])

# X, Y 분리

In [172]:
# Train
X_train = df_train.drop('Exited', axis=1)
Y_train = df_train['Exited']

# Valid
X_valid = df_valid.drop('Exited', axis=1)
Y_valid = df_valid['Exited']

# Test
X_test = df_test.drop('Exited', axis=1)
Y_test = df_test['Exited']

# Feature Transformation

In [173]:
# X 변수
X_train.isna().sum()

CreditScore            0
Geography             25
Gender                 0
Age                    0
Balance                0
NumOfProducts          0
HasCrCard              0
IsActiveMember         0
EstimatedSalary       47
TenureDays             0
BalancePerProduct      0
BalanceSalaryRatio    47
dtype: int64

In [174]:
# Y 변수 -> Missing 있는 경우 Data Partition 이전 단계에서 삭제 필요
Y_train.isna().sum()

np.int64(0)

## 연속형 변수 대체

In [175]:
# 연속형 변수 추출
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# X_train에서 중앙값 계산
medians = X_train[num_cols].median()

# Missing 값을 중앙값으로 대체
X_train[num_cols] = X_train[num_cols].fillna(medians)
X_valid[num_cols] = X_valid[num_cols].fillna(medians)
X_test[num_cols] = X_test[num_cols].fillna(medians)

## 범주형 변수 대체

In [176]:
# 범주형 변수 추출 (object + category)
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    # category 타입이면 Unknown 먼저 추가
    if X_train[col].dtype.name == 'category':
        X_train[col] = X_train[col].cat.add_categories('Unknown')
        X_valid[col] = X_valid[col].cat.add_categories('Unknown')
        X_test[col] = X_test[col].cat.add_categories('Unknown')

    # Missing → 'Unknown'
    X_train[col] = X_train[col].fillna('Unknown')
    X_valid[col] = X_valid[col].fillna('Unknown')
    X_test[col] = X_test[col].fillna('Unknown')

/var/folders/14/gczk6h2n7nb_w800tdh9mtb40000gn/T/ipykernel_81423/1492650353.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object', 'category']).columns


In [177]:
# 재확인
X_train.isnull().sum()

CreditScore           0
Geography             0
Gender                0
Age                   0
Balance               0
NumOfProducts         0
HasCrCard             0
IsActiveMember        0
EstimatedSalary       0
TenureDays            0
BalancePerProduct     0
BalanceSalaryRatio    0
dtype: int64

In [178]:
# ==========================================
# 변수 분리
# ==========================================

# 연속형 변수 추출
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# 범주형 변수 추출
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


# ==========================================
# Log Transformation 함수 정의
# ==========================================

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


# ==========================================
# Train, Valid, Test 데이터 변환 적용
# ==========================================

# Train 연속형 변수 변환
train_log = X_train[num_cols].apply(signed_log1p)

# Valid 연속형 변수 변환
valid_log = X_valid[num_cols].apply(signed_log1p)

# Test 연속형 변수 변환
test_log = X_test[num_cols].apply(signed_log1p)


# ==========================================
# 데이터 병합 (변환된 변수 + 범주형)
# ==========================================

# X_train
X_train = pd.concat(
    [
        train_log,                # Log 변환된 연속형 변수들
        X_train[cat_cols]         # 범주형 변수들
    ],
    axis=1
)

# X_valid
X_valid = pd.concat(
    [
        valid_log,                # Log 변환된 연속형 변수들
        X_valid[cat_cols]         # 범주형 변수들
    ],
    axis=1
)

# X_test
X_test = pd.concat(
    [
        test_log,                 # Log 변환된 연속형 변수들
        X_test[cat_cols]          # 범주형 변수들
    ],
    axis=1
)

/var/folders/14/gczk6h2n7nb_w800tdh9mtb40000gn/T/ipykernel_81423/73939336.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


## 데이터 스케일링(Min,Max)

In [179]:
from sklearn.preprocessing import MinMaxScaler

# 연속형 / 범주형 구분
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# Min-Max Scaler (Train에서만 fit)
scaler = MinMaxScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_valid[num_cols] = scaler.transform(X_valid[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

# 연속형 + 범주형 병합
X_train = pd.concat([X_train[num_cols], X_train[cat_cols]], axis=1)
X_valid = pd.concat([X_valid[num_cols], X_valid[cat_cols]], axis=1)
X_test  = pd.concat([X_test[num_cols],  X_test[cat_cols]], axis=1)

/var/folders/14/gczk6h2n7nb_w800tdh9mtb40000gn/T/ipykernel_81423/2911694055.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


## 범주형 변수 인코딩

In [180]:
# 범주형 변수 자동 식별
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print("\n[문자형 컬럼]:", cat_cols)

# 각 변수의 고유값 개수
print("\n[변수별 범주 개수]")
for col in cat_cols:
    n_unique = X_train[col].nunique()
    print(f"  {col}: {n_unique}개")


[문자형 컬럼]: ['Geography', 'Gender']

[변수별 범주 개수]
  Geography: 4개
  Gender: 2개


/var/folders/14/gczk6h2n7nb_w800tdh9mtb40000gn/T/ipykernel_81423/258138321.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


## One-Hot 인코딩

In [181]:
# 범주형 컬럼 선택 (train 기준)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# One-Hot Encoding
X_train = pd.get_dummies(X_train, drop_first=False, dummy_na=False)
X_valid = pd.get_dummies(X_valid, drop_first=False, dummy_na=False)
X_test  = pd.get_dummies(X_test,  drop_first=False, dummy_na=False)

# valid/test 를 train 컬럼 구조에 맞추기
#    - train에 있고, valid/test에 없는 컬럼 → 0으로 채움
#    - train에 없고, valid/test에 있는 컬럼 → 제거
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test  = X_test.reindex(columns=X_train.columns,  fill_value=0)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test  shape:", X_test.shape)

X_train shape: (1260, 16)
X_valid shape: (420, 16)
X_test  shape: (420, 16)


/var/folders/14/gczk6h2n7nb_w800tdh9mtb40000gn/T/ipykernel_81423/1445671592.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


# Feature Selection

## 단일값 변수 제거

In [182]:
# 고유값 개수 계산 (NaN 포함)
unique_counts = X_train.nunique(dropna=False)

# 고유값이 1개 이하인 컬럼 식별
cols_to_drop = unique_counts[unique_counts <= 1].index.tolist()

# 데이터셋에서 삭제
X_train = X_train.drop(columns=cols_to_drop)
X_valid = X_valid.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"삭제된 컬럼: {cols_to_drop}")

삭제된 컬럼: []


## 상관 분석

In [183]:
# 연속형 변수 컬럼 감지
continuous_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# 상관행렬 계산
corr_matrix = X_train[continuous_cols].corr()

# 상삼각(Upper Triangle) 행렬의 인덱스 추출 (대각선 제외, 중복 제거)
upper_idx = np.triu_indices_from(corr_matrix, k=1)

# 변수1, 변수2, 상관계수 형태의 DataFrame 생성
corr_df = pd.DataFrame({
    '변수1': corr_matrix.columns[upper_idx[0]],
    '변수2': corr_matrix.columns[upper_idx[1]],
    '상관계수': corr_matrix.values[upper_idx]
})

# 상관계수 절대값 기준 내림차순 정렬
corr_df = corr_df.reindex(corr_df['상관계수'].abs().sort_values(ascending=False).index)

# 결과 출력
print("=== 연속형 변수 상관분석 (상관계수 높은 순) ===")
print(corr_df.to_string(index=False))

=== 연속형 변수 상관분석 (상관계수 높은 순) ===
              변수1                변수2      상관계수
          Balance  BalancePerProduct  0.998687
BalancePerProduct BalanceSalaryRatio  0.640044
          Balance BalanceSalaryRatio  0.639284
  EstimatedSalary BalanceSalaryRatio -0.626764
    NumOfProducts  BalancePerProduct -0.390011
          Balance      NumOfProducts -0.351326
    NumOfProducts BalanceSalaryRatio -0.245379
              Age     IsActiveMember  0.089277
       TenureDays  BalancePerProduct -0.067396
          Balance         TenureDays -0.067010
   IsActiveMember         TenureDays -0.063677
              Age      NumOfProducts -0.055577
  EstimatedSalary  BalancePerProduct -0.045668
          Balance    EstimatedSalary -0.044273
    NumOfProducts    EstimatedSalary  0.038373
      CreditScore                Age -0.035970
       TenureDays BalanceSalaryRatio -0.035804
    NumOfProducts         TenureDays  0.033477
      CreditScore          HasCrCard  0.032252
    NumOfProducts          H

## 상관계수 높은 변수 제거

In [184]:
import numpy as np
import pandas as pd

# 연속형/범주형 컬럼 구분
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("최초 X변수 수:", X_train.shape[1])

# Train 기준 상관분석 → 제거할 변수 선정
corr = X_train[num_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# 상관계수 임계값
threshold = 1.0   # max: 1.0
to_drop = [col for col in upper.columns if (upper[col] >= threshold).any()]

print("제거된 연속형 변수 리스트:")
print(to_drop)

# Train/Valid/Test 동일하게 제거
X_train = X_train.drop(columns=to_drop, errors="ignore")
X_valid = X_valid.drop(columns=to_drop, errors="ignore")
X_test = X_test.drop(columns=to_drop, errors="ignore")

print("제거 후 X변수 수:", X_train.shape[1])

최초 X변수 수: 16
제거된 연속형 변수 리스트:
[]
제거 후 X변수 수: 16


## Feature Importance

In [185]:
from sklearn.ensemble import RandomForestClassifier

# RandomForest 모델 학습
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_train, Y_train)

# Feature Importance 추출
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\n[Random Forest 기준 Feature Importance]")
print(importance_df.to_string(index=False))

print("\n[총 feature 갯수]", len(X_train.columns))


[Random Forest 기준 Feature Importance]
           feature  importance
               Age    0.207377
     NumOfProducts    0.137105
       CreditScore    0.107124
   EstimatedSalary    0.101512
        TenureDays    0.096838
 BalancePerProduct    0.072058
           Balance    0.068766
BalanceSalaryRatio    0.066395
    IsActiveMember    0.055399
         HasCrCard    0.015835
 Geography_Germany    0.015022
  Geography_France    0.013723
   Geography_Spain    0.013208
     Gender_Female    0.013106
       Gender_Male    0.012956
 Geography_Unknown    0.003577

[총 feature 갯수] 16


In [186]:
# 중요도 상위 Feature 선택

print("\n[총 feature 갯수]")
max_features = len(X_train.columns)
print(f"개수: {max_features}")

TOP_N = max_features    # 원하는 상위 Feature 개수로 변경, 선택 안할 경우: max_features
final_features = importance_df.loc[:TOP_N-1, "feature"].tolist()

print(f"\n[최종 선택 변수]")
print(f"개수: {len(final_features)}")
print(f"변수: {final_features}")

# 최종 X 데이타
X_train = X_train[final_features]
X_valid = X_valid[final_features]
X_test = X_test[final_features]


[총 feature 갯수]
개수: 16

[최종 선택 변수]
개수: 16
변수: ['Age', 'NumOfProducts', 'CreditScore', 'EstimatedSalary', 'TenureDays', 'BalancePerProduct', 'Balance', 'BalanceSalaryRatio', 'IsActiveMember', 'HasCrCard', 'Geography_Germany', 'Geography_France', 'Geography_Spain', 'Gender_Female', 'Gender_Male', 'Geography_Unknown']


# 하이퍼파라미터 최적화

## Bayesian Search

In [187]:
!pip install scikit-optimize lightgbm catboost

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    KFold,
    TimeSeriesSplit
)

from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)


# ==========================================
# 기본 모델 설정
# ==========================================

# Logistic Regression
logistic_model = LogisticRegression(
    random_state=42,
    class_weight="balanced",
    max_iter=2000
)


# Random Forest
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)


# LightGBM
lgbm_model = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
    verbosity=-1
)


# CatBoost
catboost_model = CatBoostClassifier(
    random_seed=42,
    auto_class_weights="Balanced",
    verbose=0,
    thread_count=-1
)


# ==========================================
# 탐색 공간 정의
# ==========================================

# Logistic Regression
logistic_search_space = {
    "C": Real(
        0.001,
        100.0,
        prior="log-uniform"
    )
}


# Random Forest
rf_search_space = {
    "n_estimators": Integer(
        100,
        500
    ),

    "max_depth": Integer(
        2,
        20
    ),

    "min_samples_split": Integer(
        2,
        30
    ),

    "min_samples_leaf": Integer(
        1,
        20
    )
}


# LightGBM
lgbm_search_space = {
    "n_estimators": Integer(
        100,
        500
    ),

    "learning_rate": Real(
        0.01,
        0.2,
        prior="log-uniform"
    ),

    "max_depth": Integer(
        2,
        12
    ),

    "num_leaves": Integer(
        10,
        60
    )
}


# CatBoost
catboost_search_space = {
    "iterations": Integer(
        100,
        500
    ),

    "learning_rate": Real(
        0.01,
        0.2,
        prior="log-uniform"
    ),

    "depth": Integer(
        3,
        10
    ),

    "l2_leaf_reg": Real(
        1.0,
        10.0
    )
}


# ==========================================
# Cross Validation 정의
# ==========================================

# 분류 모델에서 클래스 비율 유지
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# 일반 K-Fold
# cv = KFold(
#     n_splits=3,
#     shuffle=True,
#     random_state=42
# )

# 시계열 데이터
# cv = TimeSeriesSplit(
#     n_splits=3
# )


# ==========================================
# Bayesian Search 함수
# ==========================================

def bayesian_search(
    model,
    search_space,
    model_name,
    X_train,
    Y_train
):

    print("\n")
    print("=" * 60)
    print(f"{model_name} Bayesian Search")
    print("=" * 60)

    opt = BayesSearchCV(
        estimator=model,
        search_spaces=search_space,

        # 탐색 횟수
        n_iter=15,

        # 최적 모델 판단 기준
        scoring="roc_auc",

        cv=cv,

        n_jobs=-1,

        random_state=42,

        # 최적 파라미터로
        # 전체 X_train 재학습
        refit=True,

        verbose=0
    )

    opt.fit(
        X_train,
        Y_train
    )

    print(f"\n[{model_name} CV Result]")
    print(
        "Best params:",
        opt.best_params_
    )

    print(
        "Best CV score:",
        round(opt.best_score_, 4)
    )

    return opt


# ==========================================
# Logistic Regression Bayesian Search
# ==========================================

logistic_opt = bayesian_search(
    logistic_model,
    logistic_search_space,
    "Logistic Regression",
    X_train,
    Y_train
)

best_logistic_model = (
    logistic_opt.best_estimator_
)


# ==========================================
# Random Forest Bayesian Search
# ==========================================

rf_opt = bayesian_search(
    rf_model,
    rf_search_space,
    "Random Forest",
    X_train,
    Y_train
)

best_rf_model = (
    rf_opt.best_estimator_
)


# ==========================================
# LightGBM Bayesian Search
# ==========================================

lgbm_opt = bayesian_search(
    lgbm_model,
    lgbm_search_space,
    "LightGBM",
    X_train,
    Y_train
)

best_lgbm_model = (
    lgbm_opt.best_estimator_
)


# ==========================================
# CatBoost Bayesian Search
# ==========================================

catboost_opt = bayesian_search(
    catboost_model,
    catboost_search_space,
    "CatBoost",
    X_train,
    Y_train
)

best_catboost_model = (
    catboost_opt.best_estimator_
)


# ==========================================
# 모델 평가 함수
# ==========================================

def evaluate(
    model,
    X,
    Y_actual,
    model_name,
    dataset_name
):

    # 클래스 예측
    Y_pred = model.predict(X)

    # 확률 예측
    Y_prob = model.predict_proba(X)[:, 1]

    accuracy = accuracy_score(
        Y_actual,
        Y_pred
    )

    f1 = f1_score(
        Y_actual,
        Y_pred
    )

    recall = recall_score(
        Y_actual,
        Y_pred
    )

    precision = precision_score(
        Y_actual,
        Y_pred
    )

    auc = roc_auc_score(
        Y_actual,
        Y_prob
    )

    print(
        f"\n=== {model_name} - "
        f"{dataset_name} 평가 ==="
    )

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"F1 Score : {f1:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"AUC      : {auc:.4f}"
    )

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy,
        "F1": f1,
        "Recall": recall,
        "Precision": precision,
        "AUC": auc
    }


# ==========================================
# Validation 평가
# ==========================================

validation_results = []


validation_results.append(
    evaluate(
        best_logistic_model,
        X_valid,
        Y_valid,
        "Logistic Regression",
        "Validation"
    )
)


validation_results.append(
    evaluate(
        best_rf_model,
        X_valid,
        Y_valid,
        "Random Forest",
        "Validation"
    )
)


validation_results.append(
    evaluate(
        best_lgbm_model,
        X_valid,
        Y_valid,
        "LightGBM",
        "Validation"
    )
)


validation_results.append(
    evaluate(
        best_catboost_model,
        X_valid,
        Y_valid,
        "CatBoost",
        "Validation"
    )
)


# ==========================================
# Validation 결과 비교
# ==========================================

validation_df = pd.DataFrame(
    validation_results
)

validation_df = validation_df.sort_values(
    by="AUC",
    ascending=False
)

print("\n")
print("=" * 60)
print("Validation 모델 성능 비교")
print("=" * 60)

print(
    validation_df[
        [
            "Model",
            "Accuracy",
            "F1",
            "Recall",
            "Precision",
            "AUC"
        ]
    ].round(4)
)


# ==========================================
# 가장 좋은 모델 확인
# ==========================================

best_model_name = (
    validation_df.iloc[0]["Model"]
)

print(
    f"\nValidation 기준 최고 모델: "
    f"{best_model_name}"
)


# ==========================================
# Test 평가
# ==========================================

test_results = []


test_results.append(
    evaluate(
        best_logistic_model,
        X_test,
        Y_test,
        "Logistic Regression",
        "Test"
    )
)


test_results.append(
    evaluate(
        best_rf_model,
        X_test,
        Y_test,
        "Random Forest",
        "Test"
    )
)


test_results.append(
    evaluate(
        best_lgbm_model,
        X_test,
        Y_test,
        "LightGBM",
        "Test"
    )
)


test_results.append(
    evaluate(
        best_catboost_model,
        X_test,
        Y_test,
        "CatBoost",
        "Test"
    )
)


# ==========================================
# Test 결과 비교
# ==========================================

test_df = pd.DataFrame(
    test_results
)

test_df = test_df.sort_values(
    by="AUC",
    ascending=False
)

print("\n")
print("=" * 60)
print("Test 모델 성능 비교")
print("=" * 60)

print(
    test_df[
        [
            "Model",
            "Accuracy",
            "F1",
            "Recall",
            "Precision",
            "AUC"
        ]
    ].round(4)
)



Logistic Regression Bayesian Search

[Logistic Regression CV Result]
Best params: OrderedDict([('C', 80.00069562111437)])
Best CV score: 0.7596


Random Forest Bayesian Search

[Random Forest CV Result]
Best params: OrderedDict([('max_depth', 7), ('min_samples_leaf', 1), ('min_samples_split', 30), ('n_estimators', 157)])
Best CV score: 0.8374


LightGBM Bayesian Search

[LightGBM CV Result]
Best params: OrderedDict([('learning_rate', 0.01091482290787661), ('max_depth', 2), ('n_estimators', 489), ('num_leaves', 44)])
Best CV score: 0.844


CatBoost Bayesian Search

[CatBoost CV Result]
Best params: OrderedDict([('depth', 3), ('iterations', 427), ('l2_leaf_reg', 7.671544699312246), ('learning_rate', 0.021437923903351957)])
Best CV score: 0.8581

=== Logistic Regression - Validation 평가 ===
Accuracy : 0.7071
F1 Score : 0.5654
Recall   : 0.7547
Precision: 0.4520
AUC      : 0.7931

=== Random Forest - Validation 평가 ===
Accuracy : 0.8071
F1 Score : 0.6611
Recall   : 0.7453
Precision: 0.5940

## Grid Search

In [188]:
!pip install lightgbm catboost

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)


# ==========================================
# 기본 모델 설정
# ==========================================

# Logistic Regression
logistic_model = LogisticRegression(
    random_state=42,
    class_weight="balanced",
    max_iter=2000
)


# Random Forest
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)


# LightGBM
lgbm_model = LGBMClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
    verbosity=-1
)


# CatBoost
catboost_model = CatBoostClassifier(
    random_seed=42,
    auto_class_weights="Balanced",
    verbose=0,
    thread_count=-1
)


# ==========================================
# Grid Search 탐색 공간 정의
# ==========================================

# Logistic Regression
logistic_param_grid = {
    "C": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}


# Random Forest
rf_param_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],

    "max_depth": [
        5,
        10,
        20
    ],

    "min_samples_split": [
        2,
        10,
        20
    ],

    "min_samples_leaf": [
        1,
        5,
        10
    ]
}


# LightGBM
lgbm_param_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],

    "learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "max_depth": [
        3,
        6,
        10
    ],

    "num_leaves": [
        15,
        31,
        50
    ]
}


# CatBoost
catboost_param_grid = {
    "iterations": [
        100,
        300,
        500
    ],

    "learning_rate": [
        0.01,
        0.05,
        0.1
    ],

    "depth": [
        4,
        6,
        8
    ],

    "l2_leaf_reg": [
        1,
        3,
        5
    ]
}


# ==========================================
# Cross Validation 정의
# ==========================================

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


# ==========================================
# Grid Search 함수
# ==========================================

def grid_search(
    model,
    param_grid,
    model_name,
    X_train,
    Y_train
):

    print("\n")
    print("=" * 60)
    print(f"{model_name} Grid Search")
    print("=" * 60)

    search = GridSearchCV(
        estimator=model,

        param_grid=param_grid,

        # 최적 모델 판단 기준
        scoring="roc_auc",

        cv=cv,

        n_jobs=-1,

        # 최적 파라미터로 전체 X_train 재학습
        refit=True,

        verbose=0
    )

    search.fit(
        X_train,
        Y_train
    )

    print(f"\n[{model_name} CV Result]")

    print(
        "Best params:",
        search.best_params_
    )

    print(
        "Best CV score:",
        round(search.best_score_, 4)
    )

    return search


# ==========================================
# Logistic Regression Grid Search
# ==========================================

logistic_search = grid_search(
    logistic_model,
    logistic_param_grid,
    "Logistic Regression",
    X_train,
    Y_train
)

best_logistic_model = (
    logistic_search.best_estimator_
)


# ==========================================
# Random Forest Grid Search
# ==========================================

rf_search = grid_search(
    rf_model,
    rf_param_grid,
    "Random Forest",
    X_train,
    Y_train
)

best_rf_model = (
    rf_search.best_estimator_
)


# ==========================================
# LightGBM Grid Search
# ==========================================

lgbm_search = grid_search(
    lgbm_model,
    lgbm_param_grid,
    "LightGBM",
    X_train,
    Y_train
)

best_lgbm_model = (
    lgbm_search.best_estimator_
)


# ==========================================
# CatBoost Grid Search
# ==========================================

catboost_search = grid_search(
    catboost_model,
    catboost_param_grid,
    "CatBoost",
    X_train,
    Y_train
)

best_catboost_model = (
    catboost_search.best_estimator_
)


# ==========================================
# 모델 평가 함수
# ==========================================

def evaluate(
    model,
    X,
    Y_actual,
    model_name,
    dataset_name
):

    Y_pred = model.predict(X)

    Y_prob = model.predict_proba(X)[:, 1]

    accuracy = accuracy_score(
        Y_actual,
        Y_pred
    )

    f1 = f1_score(
        Y_actual,
        Y_pred
    )

    recall = recall_score(
        Y_actual,
        Y_pred
    )

    precision = precision_score(
        Y_actual,
        Y_pred
    )

    auc = roc_auc_score(
        Y_actual,
        Y_prob
    )

    print(
        f"\n=== {model_name} - "
        f"{dataset_name} 평가 ==="
    )

    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"AUC      : {auc:.4f}")

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy,
        "F1": f1,
        "Recall": recall,
        "Precision": precision,
        "AUC": auc
    }


# ==========================================
# Validation 평가
# ==========================================

validation_results = []


validation_results.append(
    evaluate(
        best_logistic_model,
        X_valid,
        Y_valid,
        "Logistic Regression",
        "Validation"
    )
)


validation_results.append(
    evaluate(
        best_rf_model,
        X_valid,
        Y_valid,
        "Random Forest",
        "Validation"
    )
)


validation_results.append(
    evaluate(
        best_lgbm_model,
        X_valid,
        Y_valid,
        "LightGBM",
        "Validation"
    )
)


validation_results.append(
    evaluate(
        best_catboost_model,
        X_valid,
        Y_valid,
        "CatBoost",
        "Validation"
    )
)


# ==========================================
# Validation 결과 비교
# ==========================================

validation_df = pd.DataFrame(
    validation_results
)

validation_df = validation_df.sort_values(
    by="AUC",
    ascending=False
)

print("\n")
print("=" * 60)
print("Validation 모델 성능 비교")
print("=" * 60)

print(
    validation_df[
        [
            "Model",
            "Accuracy",
            "F1",
            "Recall",
            "Precision",
            "AUC"
        ]
    ].round(4)
)


# ==========================================
# Validation 기준 최고 모델
# ==========================================

best_model_name = (
    validation_df.iloc[0]["Model"]
)

print(
    f"\nValidation 기준 최고 모델: "
    f"{best_model_name}"
)


# ==========================================
# Test 평가
# ==========================================

test_results = []


test_results.append(
    evaluate(
        best_logistic_model,
        X_test,
        Y_test,
        "Logistic Regression",
        "Test"
    )
)


test_results.append(
    evaluate(
        best_rf_model,
        X_test,
        Y_test,
        "Random Forest",
        "Test"
    )
)


test_results.append(
    evaluate(
        best_lgbm_model,
        X_test,
        Y_test,
        "LightGBM",
        "Test"
    )
)


test_results.append(
    evaluate(
        best_catboost_model,
        X_test,
        Y_test,
        "CatBoost",
        "Test"
    )
)


# ==========================================
# Test 결과 비교
# ==========================================

test_df = pd.DataFrame(
    test_results
)

test_df = test_df.sort_values(
    by="AUC",
    ascending=False
)

print("\n")
print("=" * 60)
print("Test 모델 성능 비교")
print("=" * 60)

print(
    test_df[
        [
            "Model",
            "Accuracy",
            "F1",
            "Recall",
            "Precision",
            "AUC"
        ]
    ].round(4)
)



Logistic Regression Grid Search

[Logistic Regression CV Result]
Best params: {'C': 100}
Best CV score: 0.7594


Random Forest Grid Search

[Random Forest CV Result]
Best params: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}
Best CV score: 0.8401


LightGBM Grid Search

[LightGBM CV Result]
Best params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'num_leaves': 15}
Best CV score: 0.8517


CatBoost Grid Search

[CatBoost CV Result]
Best params: {'depth': 4, 'iterations': 100, 'l2_leaf_reg': 3, 'learning_rate': 0.1}
Best CV score: 0.8571

=== Logistic Regression - Validation 평가 ===
Accuracy : 0.7071
F1 Score : 0.5654
Recall   : 0.7547
Precision: 0.4520
AUC      : 0.7935

=== Random Forest - Validation 평가 ===
Accuracy : 0.7976
F1 Score : 0.6614
Recall   : 0.7830
Precision: 0.5724
AUC      : 0.8581

=== LightGBM - Validation 평가 ===
Accuracy : 0.8024
F1 Score : 0.6770
Recall   : 0.8208
Precision: 0.5762
AUC      : 0.8707

=== CatBoos

## CatBoost + LightGBM 앙상블 테스트

In [189]:
import numpy as np
import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)


# ==========================================
# 최적 하이퍼파라미터 기반 모델 정의
# ==========================================

# LightGBM
best_lgbm_model = LGBMClassifier(
    learning_rate=0.01696588362258629,
    max_depth=2,
    n_estimators=100,
    num_leaves=10,

    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
    verbosity=-1
)


# CatBoost
best_catboost_model = CatBoostClassifier(
    depth=3,
    iterations=306,
    l2_leaf_reg=1.0685432273330233,
    learning_rate=0.01,

    random_seed=42,
    auto_class_weights="Balanced",
    verbose=0,
    thread_count=-1
)


# ==========================================
# Soft Voting Ensemble 정의
# ==========================================

ensemble_model = VotingClassifier(
    estimators=[
        ("lgbm", best_lgbm_model),
        ("catboost", best_catboost_model)
    ],

    # 확률 기반 앙상블
    voting="soft",

    n_jobs=-1
)


# ==========================================
# Ensemble Grid Search 탐색 공간
# ==========================================

param_grid = {

    # LightGBM : CatBoost 가중치
    "weights": [
        (1, 1),

        (1, 2),
        (1, 3),
        (1, 4),

        (2, 1),
        (3, 1),
        (4, 1),

        (2, 3),
        (3, 2),

        (3, 4),
        (4, 3)
    ]
}


# ==========================================
# Cross Validation 정의
# ==========================================

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


# ==========================================
# Ensemble Grid Search
# ==========================================

ensemble_search = GridSearchCV(
    estimator=ensemble_model,

    param_grid=param_grid,

    # 기존 모델과 동일하게 AUC 기준
    scoring="roc_auc",

    cv=cv,

    n_jobs=-1,

    # 최적 weight로 전체 X_train 재학습
    refit=True,

    verbose=0
)


# ==========================================
# Ensemble 학습
# ==========================================

ensemble_search.fit(
    X_train,
    Y_train
)


best_ensemble_model = (
    ensemble_search.best_estimator_
)


print("\n")
print("=" * 60)
print("LightGBM + CatBoost Ensemble Grid Search")
print("=" * 60)

print(
    "Best weights:",
    ensemble_search.best_params_
)

print(
    "Best CV score:",
    round(
        ensemble_search.best_score_,
        4
    )
)


# ==========================================
# Grid Search 전체 결과 확인
# ==========================================

ensemble_cv_results = pd.DataFrame(
    ensemble_search.cv_results_
)

ensemble_cv_results = ensemble_cv_results[
    [
        "param_weights",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    by="rank_test_score"
)


print("\n")
print("=" * 60)
print("Ensemble Weight 성능 비교")
print("=" * 60)

print(
    ensemble_cv_results.round(4)
)


# ==========================================
# 평가 함수
# ==========================================

def evaluate(
    model,
    X,
    Y_actual,
    model_name,
    dataset_name
):

    Y_pred = model.predict(X)

    Y_prob = model.predict_proba(
        X
    )[:, 1]

    accuracy = accuracy_score(
        Y_actual,
        Y_pred
    )

    f1 = f1_score(
        Y_actual,
        Y_pred
    )

    recall = recall_score(
        Y_actual,
        Y_pred
    )

    precision = precision_score(
        Y_actual,
        Y_pred
    )

    auc = roc_auc_score(
        Y_actual,
        Y_prob
    )

    print(
        f"\n=== {model_name} - "
        f"{dataset_name} 평가 ==="
    )

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"F1 Score : {f1:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"AUC      : {auc:.4f}"
    )

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy,
        "F1": f1,
        "Recall": recall,
        "Precision": precision,
        "AUC": auc
    }


# ==========================================
# Validation 평가
# ==========================================

ensemble_valid_scores = evaluate(
    best_ensemble_model,
    X_valid,
    Y_valid,
    "LightGBM + CatBoost",
    "Validation"
)


# ==========================================
# Test 평가
# ==========================================

ensemble_test_scores = evaluate(
    best_ensemble_model,
    X_test,
    Y_test,
    "LightGBM + CatBoost",
    "Test"
)



LightGBM + CatBoost Ensemble Grid Search
Best weights: {'weights': (1, 4)}
Best CV score: 0.8508


Ensemble Weight 성능 비교
   param_weights  mean_test_score  std_test_score  rank_test_score
3         (1, 4)           0.8508          0.0243                1
2         (1, 3)           0.8497          0.0248                2
1         (1, 2)           0.8487          0.0252                3
7         (2, 3)           0.8476          0.0255                4
9         (3, 4)           0.8472          0.0258                5
0         (1, 1)           0.8460          0.0264                6
10        (4, 3)           0.8445          0.0267                7
8         (3, 2)           0.8439          0.0270                8
4         (2, 1)           0.8422          0.0275                9
5         (3, 1)           0.8403          0.0285               10
6         (4, 1)           0.8391          0.0286               11

=== LightGBM + CatBoost - Validation 평가 ===
Accuracy : 0.7714
F1 Score :

# Bayesian 모델 결과

In [ ]:
from catboost import CatBoostClassifier

# 최종 선정
# OrderedDict([('depth', 3), ('iterations', 427), ('l2_leaf_reg', 7.671544699312246), ('learning_rate', 0.021437923903351957)])

# 모델 생성
model = CatBoostClassifier(
    iterations=427,               # 반복 횟수
    depth=3,                      # 트리 깊이
    learning_rate=0.021437923903351957,            # 학습률
    l2_leaf_reg=7.671544699312246,                # L2 정규화 계수

    auto_class_weights="Balanced", # 클래스 불균형 자동 보정
    random_seed=42,               # 난수 시드 고정
    verbose=0                     # 학습 로그 출력 비활성화
)

# 모델 학습
model.fit(X_train, Y_train)


# X_valid 예측
Y_valid_pred = model.predict(X_valid)
Y_valid_prob = model.predict_proba(X_valid)[:, 1]  # AUC용 확률값

# X_test 예측
Y_test_pred = model.predict(X_test)
Y_test_prob = model.predict_proba(X_test)[:, 1]    # AUC용 확률값

# 평가 함수 정의
def evaluate(Y_actual, Y_pred, Y_prob, dataset_name):
    accuracy  = accuracy_score(Y_actual, Y_pred)
    f1        = f1_score(Y_actual, Y_pred)
    recall    = recall_score(Y_actual, Y_pred)
    precision = precision_score(Y_actual, Y_pred)
    auc       = roc_auc_score(Y_actual, Y_prob)

    print(f"\n=== {dataset_name} 평가 ===")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"AUC      : {auc:.4f}")

    return {'Accuracy': accuracy, 'F1': f1, 'Recall': recall,
            'Precision': precision, 'AUC': auc}

# Validation, Test 평가 실행
valid_scores = evaluate(Y_valid, Y_valid_pred, Y_valid_prob, "Validation")
test_scores  = evaluate(Y_test,  Y_test_pred,  Y_test_prob,  "Test")


=== Validation 평가 ===
Accuracy : 0.7952
F1 Score : 0.6614
Recall   : 0.7925
Precision: 0.5676
AUC      : 0.8768

=== Test 평가 ===
Accuracy : 0.7952
F1 Score : 0.5169
Recall   : 0.6765
Precision: 0.4182
AUC      : 0.8636


# Optuna

In [222]:
!pip install optuna catboost

import optuna

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)


# ==========================================
# Cross Validation 정의
# ==========================================

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


# ==========================================
# Optuna Objective 함수 정의
# ==========================================

def objective(trial):

    # 탐색할 하이퍼파라미터
    params = {
        "iterations": trial.suggest_int(
            "iterations",
            100,
            600
        ),

        "depth": trial.suggest_int(
            "depth",
            2,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.2,
            log=True
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1.0,
            15.0
        ),

        "auto_class_weights": "Balanced",

        "random_seed": 42,

        "verbose": 0,

        "thread_count": -1
    }


    # ==========================================
    # Cross Validation AUC 저장
    # ==========================================

    auc_scores = []


    for train_idx, val_idx in cv.split(
        X_train,
        Y_train
    ):

        # pandas DataFrame / Series 기준
        X_cv_train = X_train.iloc[train_idx]
        X_cv_valid = X_train.iloc[val_idx]

        Y_cv_train = Y_train.iloc[train_idx]
        Y_cv_valid = Y_train.iloc[val_idx]


        # 모델 생성
        model = CatBoostClassifier(
            **params
        )


        # 모델 학습
        model.fit(
            X_cv_train,
            Y_cv_train
        )


        # Validation 확률 예측
        Y_cv_prob = model.predict_proba(
            X_cv_valid
        )[:, 1]


        # AUC 계산
        auc = roc_auc_score(
            Y_cv_valid,
            Y_cv_prob
        )


        auc_scores.append(
            auc
        )


    # 3-Fold 평균 AUC 반환
    return sum(auc_scores) / len(auc_scores)


# ==========================================
# Optuna Study 생성
# ==========================================

study = optuna.create_study(
    direction="maximize"
)


# ==========================================
# Optuna 최적화 실행
# ==========================================

study.optimize(
    objective,

    # 탐색 횟수
    n_trials=1000
)


# ==========================================
# 최적 하이퍼파라미터 출력
# ==========================================

print("\n")
print("=" * 60)
print("Optuna Result")
print("=" * 60)

print(
    "Best params:",
    study.best_params,
)

print(
    "Best CV AUC:",
    round(
        study.best_value,
        4
    )
)

[I 2026-09-04 15:58:17,197] A new study created in memory with name: no-name-38ef2417-d470-4b36-8b35-56db6c3d38f2
[I 2026-09-04 15:58:17,628] Trial 0 finished with value: 0.8561629082203214 and parameters: {'iterations': 571, 'depth': 3, 'learning_rate': 0.011382655558749814, 'l2_leaf_reg': 12.279936092595301}. Best is trial 0 with value: 0.8561629082203214.
[I 2026-09-04 15:58:18,868] Trial 1 finished with value: 0.8282498172819434 and parameters: {'iterations': 292, 'depth': 9, 'learning_rate': 0.0176767668421057, 'l2_leaf_reg': 13.45352431304869}. Best is trial 0 with value: 0.8561629082203214.
[I 2026-09-04 15:58:20,276] Trial 2 finished with value: 0.8327284396913998 and parameters: {'iterations': 335, 'depth': 9, 'learning_rate': 0.014952885456877348, 'l2_leaf_reg': 14.956902238759227}. Best is trial 0 with value: 0.8561629082203214.
[I 2026-09-04 15:58:20,509] Trial 3 finished with value: 0.8573246941220685 and parameters: {'iterations': 375, 'depth': 2, 'learning_rate': 0.02322



Optuna Result
Best params: {'iterations': 474, 'depth': 3, 'learning_rate': 0.018330080904038207, 'l2_leaf_reg': 1.8456884701535534}
Best CV AUC: 0.8626


## Optuna 결과

Best params: {'iterations': 467, 'depth': 3, 'learning_rate': 0.018225253899166446, 'l2_leaf_reg': 9.68522800101352}

Best params: {'iterations': 552, 'depth': 3, 'learning_rate': 0.014768215310090184, 'l2_leaf_reg': 3.0964957896305583}

Best params: {'iterations': 363, 'depth': 3, 'learning_rate': 0.017003224006305002, 'l2_leaf_reg': 12.793858901654588}

Best params: {'iterations': 474, 'depth': 3, 'learning_rate': 0.018330080904038207, 'l2_leaf_reg': 1.8456884701535534}

# Optuna 모델 결과

In [223]:
from catboost import CatBoostClassifier

# 최종 선정
# OrderedDict([('depth', 3), ('iterations', 427), ('l2_leaf_reg', 7.671544699312246), ('learning_rate', 0.021437923903351957)])

# 모델 생성
model = CatBoostClassifier(
    iterations=474,               # 반복 횟수
    depth=3,                      # 트리 깊이
    learning_rate=0.018330080904038207,            # 학습률
    l2_leaf_reg=1.8456884701535534,                # L2 정규화 계수

    auto_class_weights="Balanced", # 클래스 불균형 자동 보정
    random_seed=42,               # 난수 시드 고정
    thread_count=-1,
    verbose=0                     # 학습 로그 출력 비활성화
)

# 모델 학습
model.fit(X_train, Y_train)


# X_valid 예측
Y_valid_pred = model.predict(X_valid)
Y_valid_prob = model.predict_proba(X_valid)[:, 1]  # AUC용 확률값

# X_test 예측
Y_test_pred = model.predict(X_test)
Y_test_prob = model.predict_proba(X_test)[:, 1]    # AUC용 확률값

# 평가 함수 정의
def evaluate(Y_actual, Y_pred, Y_prob, dataset_name):
    accuracy  = accuracy_score(Y_actual, Y_pred)
    f1        = f1_score(Y_actual, Y_pred)
    recall    = recall_score(Y_actual, Y_pred)
    precision = precision_score(Y_actual, Y_pred)
    auc       = roc_auc_score(Y_actual, Y_prob)

    print(f"\n=== {dataset_name} 평가 ===")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"AUC      : {auc:.4f}")

    return {'Accuracy': accuracy, 'F1': f1, 'Recall': recall,
            'Precision': precision, 'AUC': auc}

# Validation, Test 평가 실행
valid_scores = evaluate(Y_valid, Y_valid_pred, Y_valid_prob, "Validation")
test_scores  = evaluate(Y_test,  Y_test_pred,  Y_test_prob,  "Test")


=== Validation 평가 ===
Accuracy : 0.8024
F1 Score : 0.6667
Recall   : 0.7830
Precision: 0.5804
AUC      : 0.8776

=== Test 평가 ===
Accuracy : 0.8238
F1 Score : 0.5595
Recall   : 0.6912
Precision: 0.4700
AUC      : 0.8686
